# Preliminari

Si impostano directory di lavoro e si fanno import per spark

In [ ]:
import os
from pyspark.sql import SparkSession

DATASETS_DIR = "../dataset/"

spark = (
    SparkSession.builder
    .appName("pfp")
    .getOrCreate()
)

sc = spark.sparkContext


## Pre-Processing


Creo un dataframe dal file .parquet di input.

Creo i record (rdd) come necessario dal problema, ovvero chiave dell'ordine e valore le tuple contenente id oggetto e quantità.
Questi record li chiameremo Transazioni, come suggerito dal paper PFP.

In [ ]:
sdf = spark.read.parquet(
    os.path.join(DATASETS_DIR, "online_retail.parquet")
)

transactions = (
    sdf
    .select("InvoiceNo", "StockCode", "Quantity")
    .rdd
    .map(lambda row: (row["InvoiceNo"], (row["StockCode"], row["Quantity"])))
    .groupByKey()
    .mapValues(list)
)
transactions.take(5)

### Conversione

convertiamo gli oggetti (tuple chiave e quantità) in nuovi oggetti identificati da un numero, in questo modo si può facilmente utilizzare PFP con le quantità.
Riduciamo funzionalmente il problema di tenere in considerazione le quantità al problema senza le quantità per poi tornare al problema delle quantità.

T' = T
for t in T'
    t -> t'

out = PFP(T')

reversed = revert(out)

return alpha_code(reversed)

Per fare tutto ciò innanzitutto devo prendere gli oggetti e quantità e mapparli:

In [ ]:
pairs= (
    transactions
    .flatMap(lambda x: x[1])                 # prendo tutte le tuple
    .distinct()                              # tuple uniche
    .sortBy(lambda pair: (pair[0], pair[1])) # ordine stabile
    .zipWithIndex()                          # assegna indice 0,1,2...
    .map(lambda x: (x[0], x[1] + 1))         # ((code, quantity), id)
)

print("ci sono " + str(pairs.count()) + " coppie")
pairs.take(50)


In [ ]:
# Creo la mappa di conversione da tupla a numero
conversion_map = pairs.collectAsMap()
# Lo distribuisco ai worker in broadcast
bc_map = sc.broadcast(conversion_map)
bc_map.value

In [ ]:
# Creo la mappa di conversione da tupla a numero
conversion_map = pairs.collectAsMap()
# Lo distribuisco ai worker in broadcast
bc_map = sc.broadcast(conversion_map)

# Rimappo ogni transazione
transactions_ids = transactions.mapValues(
    lambda items: [bc_map.value[item] for item in items]
)
print(transactions_ids.count())
transactions_ids.take(5)

## PFP

Iniziamo ad implementare **PFP**, definiamo una variabile **epsilon** che rappresenta la "predefined minimum support threshold"
soglia minima predefinita di supporto.
Quindi una threshold sopra la quale verrà riconosciuto un pattern e i pattern sotto questa soglia verranno scartati 

In [ ]:
epsilon = 100

item_counts = (
    transactions_ids
    .flatMap(lambda row: set(row[1]))   # ogni item contato una sola volta per transazione
    .map(lambda item: (item, 1))
    .reduceByKey(lambda a, b: a + b)
    .filter(lambda row: row[1] >= epsilon)
)
print(item_counts.count())
item_counts.take(10)


Creo la F-List che è la lista decrescente degli item (item = id_of(tuple(code, quantity)))

Poi ordino le transazione per "supporto" ovvero in base ai valori di F-List

Creo infine la Q-List tramite la quale si suddivide il calcolo tra le macchine.

In [ ]:
# creo f_list
f_list = item_counts.sortBy(
    lambda row: (row[1], row[0]),
    ascending=False
)

f_list.take(10)

In [21]:
# Creo una mappa per ordinare le transazioni, la mappa è fatta così:  item_id -> posizione nella F-list

# Base comune: item_id -> rank nella F-list
item_rank = (
    f_list
    .map(lambda row: row[0])      # item_id
    .zipWithIndex()               # item_id -> rank
    .map(lambda x: (x[0], int(x[1])))
    .persist()
)

f_rank = item_rank.collectAsMap()
# notifico i worker
bc_f_rank = sc.broadcast(f_rank)

ordered_transactions = (
    transactions_ids
    .mapValues(
        lambda items: sorted(
            # tieni item solo se item è una chiave del dizionario f_rank
            set(item for item in items if item in bc_f_rank.value),
            key=lambda item: bc_f_rank.value[item]
        )
    )
    .filter(lambda row: len(row[1]) > 0)
)
print(ordered_transactions.count())
print(ordered_transactions.take(5))


16578
[('536365', [42871, 23017]), ('536367', [9514, 9531, 22959, 22914, 20952]), ('536368', [25907]), ('536369', [9544]), ('536370', [18813, 45259])]


In [22]:
# G-List
Q = spark.sparkContext.defaultParallelism # numero di core disponibili tra tutti i worker

g_list = (
    item_rank
    .map(lambda x: (x[0], int(x[1] % Q)))   # item_id -> gid
    .collectAsMap()
)

bc_g_list = sc.broadcast(g_list)

In [23]:
#Questo è fondamentalmente il mapper del paper
def generate_group_dependent_transactions(row):
    invoice_no, items = row

    output = []
    seen_gids = set()

    # Scorro la transazione da destra verso sinistra
    for j in range(len(items) - 1, -1, -1):
        item = items[j]
        gid = bc_g_list.value.get(item)

        # Se questo gruppo non è ancora stato emesso per questa transazione
        if gid is not None and gid not in seen_gids:
            seen_gids.add(gid)

            # Emetto il prefisso fino alla posizione j inclusa
            output.append((gid, items[:j + 1]))

    return output

group_dependent_transactions = ordered_transactions.flatMap(
    generate_group_dependent_transactions
)

group_dependent_transactions.take(10)

[(14, [42871, 23017]),
 (7, [42871]),
 (3, [9514, 9531, 22959, 22914, 20952]),
 (15, [9514, 9531, 22959, 22914]),
 (14, [9514, 9531, 22959]),
 (1, [9514, 9531]),
 (0, [9514]),
 (1, [25907]),
 (7, [9544]),
 (14, [18813, 45259])]

In [25]:
# Reducer del paper
# a ogni gid associo le transazioni di cui deve occupare.
group_shards = group_dependent_transactions.groupByKey()

[(14, <pyspark.resultiterable.ResultIterable at 0x753591e42e50>),
 (7, <pyspark.resultiterable.ResultIterable at 0x75359200d690>),
 (3, <pyspark.resultiterable.ResultIterable at 0x753592ec37d0>),
 (15, <pyspark.resultiterable.ResultIterable at 0x753591e2be10>),
 (1, <pyspark.resultiterable.ResultIterable at 0x753591838b10>)]